# Generated Project Anatomy

Inspect a fresh standalone wave-equation project generated by NRPy. This notebook identifies important generated files, rebuilds from a clean state, runs once, and locates the diagnostic output.

## Table of Contents

1. Step 1: Verify the imported package and generator capability
2. Step 2: Create a fresh generation workspace
3. Step 3: Generate the Cartesian wave-equation project
4. Step 4: Inventory the generated tree
5. Step 5: Connect files to responsibilities
6. Step 6: Read the parameter file and command-line override path
7. Step 7: Inspect the `Makefile`, build, clean, and rebuild
8. Step 8: Run with a convergence-factor override and inspect diagnostics
9. Step 9: What to edit, what to regenerate, and where to go next

## Step 1: Verify the Imported Package and Generator Capability

This notebook needs an importable `nrpy` package and the Cartesian wave-equation generator module. The generator is a project entry point, so this check verifies that it is discoverable without importing it.

In [ ]:
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy

PROJECT_NAME = "wave_equation_cartesian"
GENERATOR_MODULE = "nrpy.examples.wave_equation_cartesian"

print("nrpy package:", nrpy.__file__)
generator_spec = importlib.util.find_spec(GENERATOR_MODULE)
print(f"{GENERATOR_MODULE}:", generator_spec is not None)
assert generator_spec is not None, GENERATOR_MODULE

## Step 2: Create a Fresh Generation Workspace

The generator writes `project/wave_equation_cartesian/` relative to its working directory and recreates that project directory. A fresh temporary workspace keeps existing learner projects untouched. The temporary-directory object is stored so the generated project remains available for the full notebook session.

In [ ]:
_workspace = tempfile.TemporaryDirectory(prefix="nrpy-generated-project-")
WORKSPACE = Path(_workspace.name).resolve()
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

print("workspace:", WORKSPACE)
print("project directory:", PROJECT_DIR)


def _clean_text(text):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    text = text.replace("\r", "\n")
    return "\n".join(line.rstrip() for line in text.splitlines() if line.strip())


def run_command(command, cwd, max_lines=30):
    cwd = Path(cwd)
    print(f"$ {' '.join(str(part) for part in command)}")
    print(f"cwd: {cwd}")
    completed = subprocess.run(
        command,
        cwd=cwd,
        check=False,
        capture_output=True,
        text=True,
    )
    print(f"return code: {completed.returncode}")
    output = _clean_text(completed.stdout + completed.stderr)
    if output:
        lines = output.splitlines()
        if len(lines) > max_lines:
            lines = lines[: max_lines // 2] + ["..."] + lines[-max_lines // 2 :]
        print("\n".join(lines))
    if completed.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(str(part) for part in command)}")
    return completed


def require_tool(name):
    path = shutil.which(name)
    if path is None:
        raise RuntimeError(f"required command not found: {name}")
    print(f"{name}: {path}")
    return path


def preview_file(path, max_lines=12):
    path = Path(path)
    print(path.name)
    for line in path.read_text(encoding="utf-8").splitlines()[:max_lines]:
        print(line.rstrip())

## Step 3: Generate the Cartesian Wave-Equation Project

Run the generator explicitly as a module. A successful run creates `project/wave_equation_cartesian/` under the temporary workspace.

In [ ]:
run_command([sys.executable, "-m", GENERATOR_MODULE], cwd=WORKSPACE)
assert PROJECT_DIR.is_dir(), PROJECT_DIR
print(PROJECT_DIR)

## Step 4: Inventory the Generated Tree

A generated BHaH project contains C sources, headers, a `Makefile`, a parameter file, and helper subdirectories. These files are outputs of the Python generator and can be regenerated.

In [ ]:
required_relative_paths = [
    "Makefile",
    "wave_equation_cartesian.par",
    "BHaH_defines.h",
    "BHaH_function_prototypes.h",
    "main.c",
    "rhs_eval.c",
    "initial_data.c",
    "diagnostics.c",
    "numerical_grids_and_timestep.c",
    "cmdline_input_and_parfile_parser.c",
    "MoL/MoL_step_forward_in_time.c",
]

missing = [relpath for relpath in required_relative_paths if not (PROJECT_DIR / relpath).is_file()]
if missing:
    raise FileNotFoundError(f"missing generated file(s): {missing}")

c_sources = sorted(PROJECT_DIR.rglob("*.c"))
headers = sorted(PROJECT_DIR.rglob("*.h"))
print(f"C sources: {len(c_sources)}")
print(f"Headers: {len(headers)}")

print("Required files:")
for relpath in required_relative_paths:
    print(f"  {relpath}")

print("Generated file sample:")
for path in sorted(p.relative_to(PROJECT_DIR) for p in PROJECT_DIR.rglob("*") if p.is_file())[:25]:
    print(f"  {path}")

## Step 5: Connect Files to Responsibilities

The project directory is generated output. Generated C and header files are not hand-edited tutorial source; change the Python generator when the generated implementation must change, then regenerate the project.

In [ ]:
responsibilities = {
    "Makefile": "build rules for the standalone executable",
    "wave_equation_cartesian.par": "runtime parameters and defaults",
    "main.c": "program setup, time loop, diagnostics, and cleanup",
    "rhs_eval.c": "right-hand-side evaluation for the wave equation",
    "initial_data.c": "initial values for evolved grid functions",
    "diagnostics.c": "runtime numerical-output writer",
    "cmdline_input_and_parfile_parser.c": "parameter-file parsing and command-line overrides",
    "MoL/MoL_step_forward_in_time.c": "method-of-lines time update",
}

for relpath, role in responsibilities.items():
    assert (PROJECT_DIR / relpath).is_file(), relpath
    print(f"{relpath}: {role}")

## Step 6: Read the Parameter File and Command-Line Override Path

Runtime parameters live in `wave_equation_cartesian.par` unless a supported command-line argument overrides them. The Cartesian wave-equation executable accepts a convergence-factor override, so `./wave_equation_cartesian 2.0` runs with `convergence_factor = 2.0`.

In [ ]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
preview_file(parfile, max_lines=16)

parser_source = (PROJECT_DIR / "cmdline_input_and_parfile_parser.c").read_text(encoding="utf-8")
for token in ["convergence_factor", "argv", "read_REAL", "strtod"]:
    assert token in parser_source, token
print("command-line override path includes convergence_factor, argv, read_REAL, and strtod")

## Step 7: Inspect the `Makefile`, Build, Clean, and Rebuild

`make` builds the standalone executable. `make clean` removes build products while leaving generated source files in place. After a run, the same target also removes diagnostic text files; Step 8 checks that with a real diagnostic file.

In [ ]:
require_tool("make")

makefile = PROJECT_DIR / "Makefile"
preview_file(makefile, max_lines=18)

run_command(["make"], cwd=PROJECT_DIR)
executable = PROJECT_DIR / PROJECT_NAME
assert executable.is_file(), executable
print("executable exists:", executable.name)

source_before_clean = PROJECT_DIR / "main.c"
run_command(["make", "clean"], cwd=PROJECT_DIR)
assert source_before_clean.is_file(), source_before_clean
assert not executable.exists(), executable
print("clean removed the executable and kept generated sources")

run_command(["make"], cwd=PROJECT_DIR)
assert executable.is_file(), executable
print("rebuilt executable:", executable.name)

## Step 8: Run with a Convergence-Factor Override and Inspect Diagnostics

Diagnostic text files provide quick evidence that the executable produced numerical output. This run uses the command-line override `2.0`, which writes `out0d-conv_factor2.00.txt`.

In [ ]:
run_command([f"./{PROJECT_NAME}", "2.0"], cwd=PROJECT_DIR, max_lines=20)

diagnostic = PROJECT_DIR / "out0d-conv_factor2.00.txt"
assert diagnostic.is_file(), diagnostic
preview_file(diagnostic, max_lines=10)

numeric_rows = [
    line
    for line in diagnostic.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
assert numeric_rows, diagnostic
print("numeric rows:", len(numeric_rows))

run_command(["make", "clean"], cwd=PROJECT_DIR)
assert source_before_clean.is_file(), source_before_clean
assert not executable.exists(), executable
assert not diagnostic.exists(), diagnostic
assert not list(PROJECT_DIR.glob("out0d-*.txt"))
print("final clean removed executable and diagnostic products")

## Step 9: What to Edit, What to Regenerate, and Where to Go Next

Treat `project/wave_equation_cartesian/` as generated output. Edit Python generator code when changing generated C or headers, regenerate the project, then rebuild with `make`. Edit the `.par` file or pass supported command-line overrides when changing runtime parameters.

Continue with:

- [NRPy2: 10-minute Overview](../1-intro/ten_minute_overview.ipynb) for the symbolic-to-code overview.
- [Wave Equation and C Code Generation](../2-basic_physics_applications/wave_equation_and_c_codegen.ipynb) for wave-equation expressions and C code generation.
- [Start to Finish: Cartesian Wave Equation](../2-basic_physics_applications/start_to_finish__wave_equation_cartesian.ipynb) for the full Cartesian wave-equation workflow.